# 2장 06. Great Expectations 데이터 검사 실습

Great Expectations, 줄여서 GE는 데이터가 약속한 규칙을 지키는지 확인하는 도구입니다. 처음에는 어렵게 느껴질 수 있지만, 이 notebook에서는 GE를 **데이터 검사표**라고 생각하면 됩니다.


## 0. Great Expectations를 처음 보는 사람을 위한 소개

Great Expectations, 줄여서 **GE**는 데이터를 검사하는 도구입니다. 공식 사이트는 GE의 오픈소스 엔진인 **GX Core**를 데이터 품질을 확인하기 위한 프레임워크라고 소개합니다.

공식 사이트: <https://greatexpectations.io/>

쉽게 말하면 GE는 데이터에 대해 이런 질문을 자동으로 확인하게 해 줍니다.

| 쉬운 질문 | GE에서 하는 일 | 이번 실습 예시 |
| --- | --- | --- |
| 이 컬럼이 있어야 하나요? | 컬럼 존재 여부 검사 | `label` 컬럼이 있는지 확인 |
| 값이 비어 있으면 안 되나요? | 결측 검사 | `heart_rate`가 비어 있지 않은지 확인 |
| 값이 정해진 목록 안에 있어야 하나요? | 허용값 검사 | `label`이 `high_risk`, `low_risk` 중 하나인지 확인 |
| 값이 정해진 범위 안에 있어야 하나요? | 범위 검사 | `oxygen_saturation`이 0부터 100 사이인지 확인 |

GE API에서 `gx.get_context(mode=...)`를 보면 `mode` 값도 나옵니다. 이번 실습에서는 `ephemeral`만 사용합니다.

| mode | 쉬운 뜻 | 이번 강의에서의 처리 |
| --- | --- | --- |
| `ephemeral` | 메모리에만 있는 임시 GE 작업 공간 | 실습에서 사용합니다. |
| `file` | 로컬 파일로 유지되는 GE 작업 공간 | 개념만 언급합니다. |
| `cloud` | GE Cloud 연결용 작업 공간 | 강의 범위 밖입니다. 현재 GX Cloud 종료로 실습하지 않습니다. |

GE에서 자주 나오는 단어는 이렇게 이해하면 됩니다.

| GE 용어 | 쉬운 뜻 |
| --- | --- |
| Expectation | 검사 항목 |
| Expectation Suite | 검사 항목 묶음, 즉 검사표 |
| Validation Result | 검사 결과표 |
| Data Docs | 검사 결과를 사람이 읽기 쉽게 보여 주는 HTML 문서 |
| Action | 검사 실패가 났을 때 알림이나 후속 처리를 자동으로 실행하는 기능 |

공식 사이트 기준으로 GE/GX Core가 제공하는 대표 기능은 다음과 같습니다.

- Python과 Jupyter notebook에서 데이터 품질 검사를 시작할 수 있습니다.
- 여러 팀이 같은 검사 기준을 공유할 수 있습니다.
- Data Docs로 검사 결과를 사람이 읽기 쉬운 문서로 보여 줄 수 있습니다.
- 검사 실패를 기준으로 알림, 차단, 후속 처리 같은 자동화를 붙일 수 있습니다.
- 여러 데이터 도구와 연동할 수 있습니다.

이번 notebook에서는 GE의 모든 기능을 다 쓰지 않습니다. 여기서는 **검사 항목을 만들고, 데이터에 적용하고, 실패한 항목을 저장 파일로 남기는 흐름**만 연습합니다.


### 따라하기 안내

**이 notebook에서 하는 일**  
Great Expectations(GE)를 처음 보는 사람도 따라올 수 있게, GE를 “데이터 검사표를 만드는 도구”로 생각하고 진행합니다.

**흐름**  
1. 어떤 검사 항목을 쓸지 봅니다.  
2. 검사할 데이터를 봅니다.  
3. 검사 항목을 실제 데이터에 적용합니다.  
4. 실패한 항목을 파일로 저장합니다.  
5. 그 실패가 모델 평가에 어떤 제한을 주는지 정리합니다.


## 1. 실행 환경과 저장 위치 확인

### 1-1. 이 notebook이 읽고 쓰는 파일 확인

먼저 이 notebook이 어떤 데이터 파일을 읽고, 검사 결과를 어디에 저장하는지 확인합니다.

이 notebook은 로컬 파일을 다시 생성하는 실습입니다. 실행하면 `artifacts/great_expectations/` 아래 파일이 갱신될 수 있습니다. 이미 준비된 파일을 보기만 한 경우와 직접 다시 만든 경우를 보고서에서 구분해야 합니다.


### 따라하기 안내: 환경 준비

**이 셀에서 하는 일**  
이 notebook이 어떤 저장소, 어떤 데이터 파일, 어떤 출력 폴더를 사용할지 정합니다.

**확인할 점**  
`dataset`은 검사할 데이터이고, `output_dir`은 검사 결과 파일이 저장될 위치입니다.

**왜 중요한가**  
GE 검사 결과는 파일로 저장됩니다. 어디에 저장되는지 알아야 나중에 다시 확인할 수 있습니다.


In [ ]:
from __future__ import annotations

import json
import sys
from dataclasses import asdict
from pathlib import Path

import pandas as pd


def find_repo_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "packages" / "ai-quality" / "src").exists() and (base / "data").exists():
            return base
    raise RuntimeError("repo root를 찾지 못했습니다.")


ROOT = find_repo_root()
PACKAGE_SRC = ROOT / "packages" / "ai-quality" / "src"
if str(PACKAGE_SRC) not in sys.path:
    sys.path.insert(0, str(PACKAGE_SRC))

DATASET_PATH = ROOT / "data" / "vital_signs_valid_degraded.csv"
OUTPUT_DIR = ROOT / "artifacts" / "great_expectations"

execution_contract = pd.DataFrame(
    [
        {"item": "repo_root", "value": str(ROOT), "qa_use": "저장 파일 경로를 나중에 다시 확인할 수 있게 남깁니다."},
        {"item": "dataset", "value": str(DATASET_PATH), "qa_use": "검증 대상 validation 데이터를 고정합니다."},
        {"item": "output_dir", "value": str(OUTPUT_DIR), "qa_use": "검사 결과 파일이 저장될 위치를 고정합니다."},
        {"item": "runtime_role", "value": "local_result_file_regeneration", "qa_use": "이미 만들어진 파일을 읽는 실습과 구분합니다."},
    ]
)

display(execution_contract)


이 출력에서 중요한 것은 “검사 대상 데이터”와 “결과 저장 위치”입니다.

## 2. GE 검사 기준 확인

### 2-1. 검사표에 들어갈 항목 확인

GE에서 `expectation`은 검사 항목입니다. 어려운 용어처럼 보이지만, “이 컬럼은 비어 있으면 안 된다”, “이 값은 0부터 100 사이여야 한다” 같은 규칙입니다.

이 셀에서는 세 가지 기준 파일을 읽습니다.

- `dataset_schema.yaml`: 필요한 컬럼 목록
- `data_quality_rules.yaml`: 값의 허용 범위와 label 기준
- `great_expectations_rules.yaml`: GE 검사 항목

검사 결과를 보기 전에 검사 기준을 먼저 봐야 합니다.


### 따라하기 안내: 설정 파일 읽기

**이 셀에서 하는 일**  
데이터 검사 기준을 읽습니다.

**확인할 점**  
세 가지 기준을 구분합니다.

- schema: 필요한 컬럼 목록
- quality rule: 값의 허용 범위와 label 값
- GE expectation: 실제 검사 항목

**왜 중요한가**  
검사는 기준이 있어야 할 수 있습니다. 이 셀은 그 기준표를 불러오는 단계입니다.


In [ ]:
from ai_quality.common.config import load_yaml
from ai_quality.common.paths import config_path
from ai_quality.data_quality.domain.dataset_schema import DatasetSchema
from ai_quality.data_quality.domain.quality_rule import DataQualityRules

schema_config = load_yaml(config_path("validation", "dataset_schema.yaml"))
rules_config = load_yaml(config_path("validation", "data_quality_rules.yaml"))
expectations_config = load_yaml(config_path("validation", "great_expectations_rules.yaml"))

schema = DatasetSchema.from_config(schema_config)
rules = DataQualityRules.from_config(rules_config)
expectations = expectations_config["expectations"]
schema_feature_columns = list(schema.model_feature_columns)

expectation_table = pd.DataFrame(expectations).assign(
    evidence_role=lambda table: table["column"].map(lambda column: "label_contract" if column == schema.target_column else "feature_quality")
)
rule_scope = pd.DataFrame(
    [
        {"rule_source": "dataset_schema.yaml", "observed": f"target={schema.target_column}, required_columns={len(schema.required_columns)}", "qa_use": "평가 입력 구조를 확인합니다."},
        {"rule_source": "data_quality_rules.yaml", "observed": f"valid_ranges={len(rules.valid_ranges)}, allowed_labels={rules.allowed_labels}", "qa_use": "feature 범위와 label 허용값을 확인합니다."},
        {"rule_source": "great_expectations_rules.yaml", "observed": f"expectations={len(expectations)}", "qa_use": "저장 파일에 남길 검증 기준을 확인합니다."},
    ]
)


### 출력 확인: `rule_scope`

**확인할 점**  
검사에 사용할 기준 파일이 3개 보입니다.

- `dataset_schema.yaml`: 어떤 컬럼이 있어야 하는지 정합니다.
- `data_quality_rules.yaml`: 값의 허용 범위와 label 값을 정합니다.
- `great_expectations_rules.yaml`: GE 검사 항목을 정합니다.

**왜 중요한가**  
검사 결과를 보기 전에 “무슨 기준으로 검사했는지”를 알아야 합니다. 기준을 모르면 실패가 왜 문제인지 설명할 수 없습니다.


In [ ]:
display(rule_scope)


### 출력 확인: `expectation_table`

**확인할 점**  
GE 검사 항목을 봅니다. 여기서 `expectation`은 어려운 말이지만, 그냥 “검사 항목”이라고 생각하면 됩니다.

예를 들어 다음과 같습니다.

- `label` 컬럼이 있어야 한다.
- `label`은 비어 있으면 안 된다.
- `heart_rate`는 비어 있으면 안 된다.
- `oxygen_saturation`은 0부터 100 사이여야 한다.

**왜 중요한가**  
이 표가 GE로 무엇을 검사할지 보여 주는 검사표입니다. 뒤에서 이 검사표를 실제 데이터에 적용해 pass/fail 결과를 만듭니다.


In [ ]:
display(expectation_table)


### 2-2. GE를 실제로 어디에서 쓰는지 확인

앞에서 본 `great_expectations_rules.yaml`은 검사 항목 목록입니다. 이제 이 검사 항목을 실제 GE 객체로 옮깁니다.

쉽게 말하면 다음 순서입니다.

1. GE 패키지를 켭니다.
2. 빈 검사표 묶음, 즉 `ExpectationSuite`를 만듭니다.
3. YAML에 적힌 검사 항목을 GE 검사 class로 바꿔서 suite에 넣습니다.

여기서 GE가 직접 하는 일은 “검사표 묶음을 만드는 것”입니다. 실제 pass/fail 계산과 파일 저장은 뒤에서 저장소 공통 코드가 이어서 처리합니다.


In [ ]:
import great_expectations as gx
from great_expectations.core.expectation_suite import ExpectationSuite
from great_expectations.expectations import (
    ExpectColumnToExist,
    ExpectColumnValuesToBeBetween,
    ExpectColumnValuesToBeInSet,
    ExpectColumnValuesToNotBeNull,
)

GE_EXPECTATION_CLASS = {
    "expect_column_to_exist": ExpectColumnToExist,
    "expect_column_values_to_not_be_null": ExpectColumnValuesToNotBeNull,
    "expect_column_values_to_be_in_set": ExpectColumnValuesToBeInSet,
    "expect_column_values_to_be_between": ExpectColumnValuesToBeBetween,
}

ge_context = gx.get_context(mode="ephemeral")
ge_suite = ExpectationSuite(name="chapter_02_validation_degraded")

ge_suite_rows = []
for item in expectations:
    expectation_type = item["expectation_type"]
    expectation_class = GE_EXPECTATION_CLASS[expectation_type]
    kwargs = {"column": item["column"]}
    if "value_set" in item:
        kwargs["value_set"] = item["value_set"]
    if "min_value" in item:
        kwargs["min_value"] = item["min_value"]
    if "max_value" in item:
        kwargs["max_value"] = item["max_value"]

    ge_expectation = expectation_class(**kwargs)
    ge_suite.add_expectation(ge_expectation)
    ge_suite_rows.append(
        {
            "expectation_type": expectation_type,
            "column": item["column"],
            "ge_class": expectation_class.__name__,
            "qa_reason": item.get("qa_reason", ""),
        }
    )

ge_usage_table = pd.DataFrame(
    [
        {"item": "great_expectations_version", "value": gx.__version__, "meaning": "실제 GE 패키지를 import했습니다."},
        {"item": "data_context", "value": type(ge_context).__name__, "meaning": "로컬 파일을 쓰지 않는 ephemeral context입니다."},
        {"item": "expectation_suite", "value": ge_suite.name, "meaning": "YAML 기준을 GE suite 객체로 옮겼습니다."},
        {"item": "suite_expectation_count", "value": len(ge_suite.expectations), "meaning": "GE suite에 들어간 expectation 수입니다."},
        {"item": "pass_fail_engine", "value": "course QualityReport -> ValidationResult", "meaning": "교육용 저장 파일은 저장소 공통 코드가 같은 기준으로 계산합니다."},
    ]
)

ge_suite_table = pd.DataFrame(ge_suite_rows)


### 출력 확인: `ge_usage_table`

**확인할 점**  
실제로 `great_expectations` 패키지가 import되었는지, GE context와 expectation suite가 만들어졌는지 봅니다.

**쉽게 말하면**  
이 표는 “GE 도구를 켰고, 검사표 묶음을 만들었다”는 확인입니다.

**주의할 점**  
여기서는 GE Cloud나 서버 화면을 띄우지 않습니다. 로컬 notebook 안에서 GE의 검사표 구조만 사용합니다.


In [ ]:
display(ge_usage_table)


### 출력 확인: `ge_suite_table`

**확인할 점**  
YAML에 적힌 검사 항목이 GE의 실제 검사 class로 바뀐 것을 봅니다.

예를 들어 다음처럼 연결됩니다.

- `expect_column_values_to_not_be_null` -> 비어 있으면 안 된다는 GE 검사
- `expect_column_values_to_be_between` -> 값이 정해진 범위 안에 있어야 한다는 GE 검사

**왜 중요한가**  
이 표를 보면 “GE가 어디에 쓰였는지”가 보입니다. 설정 파일에 적은 검사 기준을 GE 검사표 묶음으로 옮긴 단계입니다.

**다음 단계**  
이제 이 검사 기준을 실제 데이터에 적용해서 어떤 항목이 통과했고 실패했는지 봅니다.


In [ ]:
display(ge_suite_table)


이 출력에서 중요한 것은 GE 검사 항목이 “보고서 근거”로 이어진다는 점입니다. 각 항목에는 왜 이 검사가 필요한지 설명하는 `qa_reason`이 들어 있습니다.

### 2-3. 검사할 데이터 원본 확인

이제 검사할 데이터를 봅니다. 아직 검사를 실행하지 않았습니다. 먼저 행 수, 컬럼 수, label 분포, 결측 현황을 확인합니다.

원본 데이터에서 이미 보이는 문제가 GE 검사 실패로 이어지는지 확인해 봅니다.


### 따라하기 안내: 데이터 로딩

**이 셀에서 하는 일**  
검사할 degraded validation 데이터를 읽습니다.

**확인할 점**  
행 수, label 분포, 결측 현황을 봅니다.

**왜 중요한가**  
검사 결과를 보기 전에 원본 데이터에 어떤 문제가 보이는지 먼저 확인합니다.


In [ ]:
raw_dataframe = pd.read_csv(DATASET_PATH)
preview_columns = [
    column for column in ["patient_id", "timestamp", *schema_feature_columns, schema.target_column]
    if column in raw_dataframe.columns
]

raw_data_brief = pd.DataFrame(
    [
        {"check": "dataset_path", "observed": str(DATASET_PATH), "qa_use": "검증 대상 파일을 고정합니다."},
        {"check": "row_grain", "observed": "one validation classification sample", "qa_use": "검증 실패 비율의 분모를 설명합니다."},
        {"check": "row_count", "observed": raw_dataframe.shape[0], "qa_use": "기대 조건 실패 비율 계산 기준입니다."},
        {"check": "column_count", "observed": raw_dataframe.shape[1], "qa_use": "schema 검증 전 구조를 확인합니다."},
    ]
)
raw_label_distribution = (
    raw_dataframe[schema.target_column]
    .value_counts(dropna=False)
    .rename_axis("raw_label")
    .reset_index(name="count")
    .assign(ratio_pct=lambda table: (table["count"] / len(raw_dataframe) * 100).round(2))
)
raw_missing_snapshot = (
    raw_dataframe.loc[:, schema_feature_columns + [schema.target_column]]
    .isna()
    .sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "column"})
    .assign(missing_ratio_pct=lambda table: (table["missing_count"] / len(raw_dataframe) * 100).round(2))
    .sort_values(["missing_count", "column"], ascending=[False, True])
)


### 출력 확인: `raw_data_brief`

**확인할 점**  
검사할 데이터의 행 수와 컬럼 수를 봅니다.

**왜 중요한가**  
지금 어떤 데이터를 검사하는지 먼저 확인해야 합니다. 데이터가 다르면 GE 결과도 달라집니다.


In [ ]:
display(raw_data_brief)


### 출력 확인: `raw_dataframe.loc[:, preview_columns].head()`

**확인할 점**  
앞 몇 줄을 직접 보면서 컬럼 이름과 값 모양이 예상과 맞는지 확인합니다.

**왜 중요한가**  
표의 실제 모습을 보면 뒤에서 나오는 검사 실패가 어느 컬럼과 관련되는지 이해하기 쉽습니다.

**주의할 점**  
앞 몇 줄만 보고 전체 데이터를 확인하지 않습니다. 전체 결측과 범위 오류는 다음 표에서 봅니다.


In [ ]:
display(raw_dataframe.loc[:, preview_columns].head())


### 출력 확인: `raw_label_distribution`

**확인할 점**  
`high_risk`, `low_risk` label이 각각 몇 개인지 봅니다.

**왜 중요한가**  
label은 모델 평가의 정답입니다. 정답 label이 있어야 나중에 모델 예측이 맞았는지 틀렸는지 계산할 수 있습니다.


In [ ]:
display(raw_label_distribution)


### 출력 확인: `raw_missing_snapshot.head(12)`

**확인할 점**  
어떤 컬럼에 빈 값이 있는지 봅니다.

**왜 중요한가**  
빈 값이 모델 입력 컬럼에 있으면 score 계산이 흔들릴 수 있습니다. 이 실습에서는 특히 `heart_rate` 결측을 확인합니다.


In [ ]:
display(raw_missing_snapshot.head(12))


이 출력에서 중요한 것은 GE 결과가 갑자기 생긴 것이 아니라는 점입니다. 검사 전에 원본 데이터에서 이미 `heart_rate` 결측 같은 신호가 보입니다.

## 3. 데이터 검사 실행

### 3-1. 중간 품질 보고서 만들기

먼저 저장소의 공통 품질 검사 코드로 중간 보고서를 만듭니다. 이 보고서는 컬럼별 결측, 범위 오류, label 수를 계산합니다.

그 다음 단계에서 이 중간 보고서를 GE 검사 항목별 pass/fail 결과로 바꿉니다.


### 따라하기 안내: 품질 검사 실행

**이 셀에서 하는 일**  
데이터를 한 번 검사해서 중간 품질 보고서를 만듭니다.

**확인할 점**  
컬럼별 결측, 범위 오류, label 수를 나누어 봅니다.

**왜 중요한가**  
GE 결과는 이 중간 품질 보고서를 검사 항목별 pass/fail로 바꾼 것입니다.


In [ ]:
from ai_quality.data_quality.application.inspect_dataset_quality import InspectDatasetQuality
from ai_quality.data_quality.infrastructure.pandas_dataset_reader import PandasDatasetReader

quality_report = InspectDatasetQuality(
    dataset_reader=PandasDatasetReader(schema),
    schema=schema,
    rules=rules,
).run(DATASET_PATH)

column_quality = pd.DataFrame([asdict(item) for item in quality_report.column_quality]).sort_values(
    ["missing_count", "column"], ascending=[False, True]
)
range_results = pd.DataFrame([asdict(item) for item in quality_report.range_results]).sort_values(
    ["invalid_count", "column"], ascending=[False, True]
)
label_support = pd.DataFrame([asdict(quality_report.label_support)]).assign(
    positive_rate_pct=round(quality_report.label_support.positive_rate, 2),
    labeled_count=quality_report.label_support.labeled_count,
)
quality_report_summary = pd.DataFrame(
    [
        {
            "row_count": quality_report.row_count,
            "column_count": quality_report.column_count,
            "missing_columns": list(quality_report.missing_columns),
            "is_evaluation_ready": quality_report.is_evaluation_ready,
            "qa_judgment": "label 기준은 유지되지만 feature 품질 실패를 expectation 결과로 확인해야 합니다."
            if quality_report.is_evaluation_ready else "평가 전 보류 조건이 있는지 세부 실패를 확인해야 합니다.",
        }
    ]
)


### 출력 확인: `quality_report_summary`

**확인할 점**  
전체 데이터 검사 요약을 봅니다. 행 수, 컬럼 수, 평가 준비 상태를 확인합니다.

**왜 중요한가**  
이 표는 GE 결과를 만들기 전의 중간 검사표입니다. 여기서 데이터가 대체로 평가 가능한지, 그래도 제한 사항이 남는지 확인합니다.

**주의할 점**  
`is_evaluation_ready`가 좋아 보여도 모든 문제가 사라진 것은 아닙니다. feature 결측이나 범위 오류는 따로 봐야 합니다.


In [ ]:
display(quality_report_summary)


### 출력 확인: `column_quality.head(12)`

**확인할 점**  
컬럼별 결측 개수를 봅니다. 어떤 컬럼이 비어 있는지 확인합니다.

**왜 중요한가**  
결측이 label에 있으면 정답 비교가 어렵고, feature에 있으면 모델 score가 흔들릴 수 있습니다.

**다음 단계**  
결측이 있는 컬럼이 GE 검사 실패로 이어지는지 확인합니다.


In [ ]:
display(column_quality.head(12))


### 출력 확인: `range_results.loc[range_results["invalid_count"].gt(0)]`

**확인할 점**  
허용 범위를 벗어난 값이 있는 컬럼만 봅니다.

**왜 중요한가**  
예를 들어 산소포화도는 0부터 100 사이여야 합니다. 이 범위를 벗어나면 데이터 입력이나 단위 문제일 수 있고, 모델 score도 이상해질 수 있습니다.


In [ ]:
display(range_results.loc[range_results["invalid_count"].gt(0)])


### 출력 확인: `label_support`

**확인할 점**  
정답 label이 몇 개 있는지, `high_risk`와 `low_risk`가 모두 있는지 봅니다.

**왜 중요한가**  
label은 채점 기준입니다. label이 유지되면 모델 지표를 계산할 형식은 갖춘 것입니다. 다만 feature 품질 문제는 별도로 봐야 합니다.


In [ ]:
display(label_support)


이 출력에서 중요한 것은 label과 feature를 나누어 보는 것입니다.

label 기준이 유지되면 모델 지표를 계산할 형식은 갖춘 것입니다. 하지만 feature에 결측이나 범위 오류가 있으면 score와 metric 해석에 제한이 생깁니다.

### 3-2. GE 검사 항목별 pass/fail 만들기

이제 검사 항목별 결과표를 만듭니다. 이 표가 이 notebook의 핵심 결과입니다.

각 행은 이렇게 읽으면 됩니다.

```text
이 컬럼에 대해, 이 검사를 했고, 결과는 pass 또는 fail이다.
```


### 따라하기 안내: 검사 결과표 생성

**이 셀에서 하는 일**  
GE 검사 항목별로 pass/fail 결과를 만듭니다.

**확인할 점**  
어떤 검사 항목이 어떤 컬럼에서 실패했는지 봅니다.

**왜 중요한가**  
이 표가 “데이터 검사 결과표”입니다. 모델 지표를 보기 전에 어떤 데이터 문제가 있었는지 알려 줍니다.


In [ ]:
from ai_quality.data_quality.application.build_validation_result import build_validation_result

validation_result = build_validation_result(
    report=quality_report,
    expectations=expectations,
    dataset_name=DATASET_PATH.name,
)

validation_result_table = pd.DataFrame(
    [asdict(result) for result in validation_result.expectation_results]
).assign(
    status=lambda table: table["success"].map({True: "pass", False: "fail"}),
    unexpected_ratio_pct=lambda table: table["unexpected_ratio"].round(2),
    evidence_role=lambda table: table["column"].map(lambda column: "label_contract" if column == schema.target_column else "feature_quality"),
)
validation_summary = pd.DataFrame(
    [
        {
            "dataset_name": validation_result.dataset_name,
            "row_count": validation_result.row_count,
            "overall_success": validation_result.success,
            "success_count": validation_result.success_count,
            "failure_count": validation_result.failure_count,
            "qa_judgment": "feature 품질 실패가 있어 조건부 평가와 metric 해석 제한이 필요합니다."
            if validation_result.failure_count else "설정한 expectation 기준은 모두 통과했습니다.",
        }
    ]
)
failure_table = validation_result_table.loc[
    validation_result_table["status"].eq("fail"),
    ["expectation_type", "column", "status", "unexpected_count", "unexpected_ratio_pct", "observed_value", "qa_reason", "evidence_role"],
]


### 출력 확인: `validation_summary`

**확인할 점**  
GE 검사 항목 중 몇 개가 통과했고 몇 개가 실패했는지 봅니다.

**왜 중요한가**  
이 표가 “검사표를 실제 데이터에 적용한 전체 결과”입니다. 실패가 있으면 어떤 컬럼에서 실패했는지 다음 표에서 확인합니다.


In [ ]:
display(validation_summary)


### 출력 확인: `validation_result_table`

**확인할 점**  
검사 항목별로 `pass`인지 `fail`인지 봅니다.

**쉽게 읽는 법**  
한 줄을 이렇게 읽으면 됩니다.

```text
이 컬럼에 대해, 이 검사를 했고, 결과는 pass 또는 fail이다.
```

예를 들어 `heart_rate`의 not null 검사가 fail이면, `heart_rate`에 빈 값이 있다는 뜻입니다.

**왜 중요한가**  
이 표가 GE 실습의 핵심 결과입니다. 모델 평가 전에 어떤 데이터 문제가 있었는지 보여 줍니다.


In [ ]:
display(validation_result_table.loc[:, ["expectation_type", "column", "status", "unexpected_count", "unexpected_ratio_pct", "observed_value", "qa_reason", "evidence_role"]])


### 출력 확인: `failure_table`

**확인할 점**  
실패한 검사만 모아서 봅니다.

**왜 중요한가**  
전체 검사표보다 실패 목록이 더 중요합니다. 여기서 어떤 컬럼을 먼저 확인해야 하는지 정합니다.

**해석 방법**  
실패가 `label`이 아니라 feature 컬럼에서 났다면, 모델 지표 계산 형식은 유지되지만 score 해석에는 제한이 생깁니다.


In [ ]:
display(failure_table)


이 출력에서 중요한 것은 실패 위치입니다.

실패가 `label`이 아니라 feature 컬럼에서 발생했다면, 모델 지표 계산은 가능하지만 score 해석에는 제한이 생깁니다.

## 4. 검사 결과를 파일로 저장

### 4-1. GE 저장 파일 만들기

검사 결과를 화면에서만 보고 끝내지 않고 파일로 저장합니다.

생성되는 파일은 네 가지입니다.

- expectations JSON: 어떤 검사를 했는지
- 검사 결과표 JSON: 검사 결과 원자료
- summary Markdown: 보고서용 요약
- Data Docs HTML: 사람이 보기 쉬운 검사 결과 화면

이 파일들이 있어야 나중에 같은 근거를 다시 확인할 수 있습니다.


### 따라하기 안내: GE 저장 파일 저장

**이 셀에서 하는 일**  
검사 기준과 검사 결과를 파일로 저장합니다.

**확인할 점**  
JSON, Markdown, HTML 파일이 만들어졌는지 봅니다.

**왜 중요한가**  
결과가 파일로 남아야 보고서와 리뷰에서 같은 근거를 다시 확인할 수 있습니다.


In [ ]:
from ai_quality.data_quality.infrastructure.great_expectations_artifact_writer import (
    serialize_validation_result,
    write_great_expectations_demo_artifacts,
)

artifacts = write_great_expectations_demo_artifacts(
    expectations=expectations,
    validation_result=validation_result,
    output_dir=OUTPUT_DIR,
)

saved_file_table = pd.DataFrame(
    [
        {"file_kind": "expectations", "path": str(artifacts.expectations_path), "exists": artifacts.expectations_path.exists(), "qa_use": "적용한 검사 기준 확인"},
        {"file_kind": "validation_result", "path": str(artifacts.validation_result_path), "exists": artifacts.validation_result_path.exists(), "qa_use": "어떤 검사가 실패했는지 확인"},
        {"file_kind": "validation_summary", "path": str(artifacts.validation_summary_path), "exists": artifacts.validation_summary_path.exists(), "qa_use": "보고서에 쓸 요약 확인"},
        {"file_kind": "data_docs", "path": str(artifacts.data_docs_path), "exists": artifacts.data_docs_path.exists(), "qa_use": "사람이 읽기 쉬운 HTML 결과 확인"},
    ]
)
serialized_result = serialize_validation_result(validation_result)
artifact_content_check = pd.DataFrame(
    [
        {
            "check": "serialized_failure_count",
            "observed": serialized_result["failure_count"],
            "expected_use": "저장된 결과 파일과 문서 요약의 실패 조건 수와 맞아야 합니다.",
        },
        {
            "check": "serialized_row_count",
            "observed": serialized_result["row_count"],
            "expected_use": "검증 대상 데이터셋 규모와 맞아야 합니다.",
        },
    ]
)


### 출력 확인: `저장 파일_table`

**확인할 점**  
생성된 파일 경로와 `exists=True`인지 봅니다.

**왜 중요한가**  
검사 결과가 notebook 화면에만 있으면 나중에 다시 확인하기 어렵습니다. 파일로 저장되어야 보고서 근거가 됩니다.


In [ ]:
display(saved_file_table)


### 출력 확인: `저장 파일_content_check`

**확인할 점**  
저장할 결과의 실패 수와 row 수가 기대와 맞는지 봅니다.

**왜 중요한가**  
파일이 만들어졌다는 사실만으로는 부족합니다. 파일 안에 들어갈 핵심 숫자도 맞는지 확인해야 합니다.


In [ ]:
display(artifact_content_check)


이 출력에서 중요한 것은 검사 결과가 파일로 남았다는 점입니다. 파일 경로가 있어야 다음 notebook이나 보고서에서 같은 근거를 다시 열어볼 수 있습니다.

### 4-2. 저장한 파일 다시 읽기

파일을 만들었다는 사실만으로는 부족합니다. 저장한 JSON을 다시 읽어서 실패 수와 실패 항목이 그대로 들어 있는지 확인합니다.


### 따라하기 안내: 저장 결과 확인

**이 셀에서 하는 일**  
방금 저장한 JSON 파일을 다시 읽습니다.

**확인할 점**  
다시 읽어도 실패 수와 실패 항목이 그대로인지 봅니다.

**왜 중요한가**  
저장된 파일이 실제 보고서 근거로 쓸 수 있는지 확인하는 단계입니다.


In [ ]:
reloaded_validation_result = json.loads(artifacts.validation_result_path.read_text(encoding="utf-8"))
reloaded_table = pd.DataFrame(reloaded_validation_result["expectation_results"]).assign(
    status=lambda table: table["success"].map({True: "pass", False: "fail"}),
    unexpected_ratio_pct=lambda table: table["unexpected_ratio"].round(2),
)
reloaded_failures = reloaded_table.loc[
    reloaded_table["status"].eq("fail"),
    ["expectation_type", "column", "status", "unexpected_count", "unexpected_ratio_pct", "observed_value", "qa_reason"],
]
reload_check = pd.DataFrame(
    [
        {
            "check": "failure_count_matches",
            "observed": reloaded_validation_result["failure_count"],
            "expected": validation_result.failure_count,
            "status": "pass" if reloaded_validation_result["failure_count"] == validation_result.failure_count else "review",
        },
        {
            "check": "row_count_matches",
            "observed": reloaded_validation_result["row_count"],
            "expected": validation_result.row_count,
            "status": "pass" if reloaded_validation_result["row_count"] == validation_result.row_count else "review",
        },
    ]
)


### 출력 확인: `reload_check`

**확인할 점**  
저장한 JSON 파일을 다시 읽었을 때 실패 수와 row 수가 원래 결과와 같은지 봅니다.

**왜 중요한가**  
다시 읽어도 같은 값이면, 이 파일을 보고서 근거로 사용할 수 있습니다.


In [ ]:
display(reload_check)


### 출력 확인: `reloaded_failures`

**확인할 점**  
저장된 파일에서 다시 읽은 실패 목록을 봅니다.

**왜 중요한가**  
이 표는 검토자가 나중에 같은 파일을 열었을 때 확인할 실패 항목입니다.


In [ ]:
display(reloaded_failures)


이 출력에서 중요한 것은 저장한 파일을 다시 읽어도 같은 실패 항목이 보인다는 점입니다. 이제 이 파일을 보고서 근거로 사용할 수 있습니다.

## 5. QA 해석으로 정리

### 5-1. 검사 실패를 모델 평가 제한 사항으로 바꾸기

마지막으로 GE 검사 실패를 사람이 읽을 수 있는 QA 해석으로 바꿉니다.

여기서 중요한 구분은 두 가지입니다.

- label 문제가 있으면 모델 평가의 정답 기준이 흔들립니다.
- feature 문제가 있으면 모델 score와 metric 해석에 제한이 생깁니다.

이번 실습에서는 feature 품질 실패를 다음 notebook에서 score, prediction, FP/FN 변화와 연결합니다.


### 따라하기 안내: 실패 column 요약

**이 셀에서 하는 일**  
실패한 검사 결과를 QA 해석으로 바꿉니다.

**확인할 점**  
실패가 label 문제인지, feature 문제인지 구분합니다.

**왜 중요한가**  
label 문제면 모델 평가 자체가 흔들릴 수 있고, feature 문제면 score와 metric 해석에 제한이 생깁니다.


In [ ]:
failed_columns = reloaded_failures["column"].tolist()
feature_failures = reloaded_failures.loc[reloaded_failures["column"].ne(schema.target_column)]
label_failures = reloaded_failures.loc[reloaded_failures["column"].eq(schema.target_column)]

qa_decision = "conditional_evaluation" if len(feature_failures) and label_failures.empty else (
    "hold_evaluation" if not label_failures.empty else "evaluation_ready"
)

chapter_02_ge_packet = pd.DataFrame(
    [
        {
            "evidence": "ge_validation_result",
            "observed": f"failure_count={reloaded_validation_result['failure_count']}, failed_columns={failed_columns}",
            "qa_judgment": "feature 품질 실패가 있어 모델 metric 해석에 제한을 붙입니다.",
            "owner": "QA/Data Engineering",
            "next_action": "2-5에서 score/prediction/FP/FN 변화와 연결합니다.",
        },
        {
            "evidence": "label_contract",
            "observed": f"label_failures={len(label_failures)}",
            "qa_judgment": "허용 label 형식은 유지되지만 label basis review 후보는 별도로 남깁니다.",
            "owner": "Data Owner/QA",
            "next_action": "라벨 반전 후보는 metric 해석 제한 사항으로 기록합니다.",
        },
        {
            "evidence": "saved_result_file",
            "observed": str(artifacts.validation_result_path),
            "qa_judgment": "검사 결과가 다시 만들 수 있는 저장 파일로 남았습니다.",
            "owner": "QA",
            "next_action": "JupyterLite 실습에서는 이 저장 파일을 준비된 근거로 읽습니다.",
        },
    ]
)

handoff_to_data_metric = pd.DataFrame(
    [
        {
            "from_notebook": "06_great_expectations_lab.ipynb",
            "to_notebook": "08_data_metric_connection_lab.ipynb",
            "decision": qa_decision,
            "strengthened_candidates": "feature_quality_shift" if len(feature_failures) else "none",
            "open_candidates": "label_basis_review, metric_error_shift",
            "next_question": "feature 품질 실패가 score, prediction, FP/FN 변화와 함께 나타나는가?",
        }
    ]
)

report_sentence = (
    f"Great Expectations Demo 재생성 결과, {DATASET_PATH.name}에서 "
    f"failure_count={reloaded_validation_result['failure_count']}가 확인되었습니다. "
    f"실패 컬럼은 {failed_columns}이며, label 형식 실패는 {len(label_failures)}건입니다. "
    "따라서 모델 평가를 모델 문제로 단정하지 않고, feature 품질 실패가 score와 FP/FN 변화로 이어지는지 2-5에서 확인합니다."
)

print(report_sentence)


### 출력 확인: `chapter_02_ge_packet`

**확인할 점**  
GE 검사 결과를 QA 해석 문장으로 정리한 표입니다.

**왜 중요한가**  
여기서부터는 도구 결과가 아니라 보고서에 쓸 확인입니다. 어떤 실패가 있었고, 누가 다음 확인을 해야 하는지 봅니다.


In [ ]:
display(chapter_02_ge_packet)


### 출력 확인: `handoff_to_data_metric`

**확인할 점**  
다음 notebook인 데이터-지표 연결 실습으로 넘길 내용을 봅니다.

**왜 중요한가**  
GE 검사 실패만으로 모델 문제를 확정하지 않습니다. 다음 단계에서 이 실패가 score, prediction, FP/FN 변화와 연결되는지 확인합니다.


In [ ]:
display(handoff_to_data_metric)


### 5-2. 실패 시 확인 포인트

실행이 실패하면 먼저 `DATASET_PATH`와 `OUTPUT_DIR`을 확인합니다. 데이터 파일을 찾지 못하면 `labs/prepare_data.py` 또는 데이터 준비 절차가 먼저 필요합니다.

검증 결과가 문서와 다르면 `dataset_schema.yaml`, `data_quality_rules.yaml`, `great_expectations_rules.yaml`이 바뀌었는지 확인합니다. 규칙이 바뀌면 실패 수가 달라지는 것은 정상일 수 있지만, 보고서에는 새 기준으로 재생성했다고 적어야 합니다.

저장 파일이 생성되지 않으면 `artifacts/great_expectations` 쓰기 권한과 경로를 확인합니다. 이 notebook은 일반 로컬 재생성 경로이므로 JupyterLite에서 실행할 대상이 아닙니다.
